## Dashboard data

**Purpose:** create the tables used by the Power BI dashboard from the raw readings and the outputs of the K-means notebook.

**Inputs:** the raw data file, `home_features.csv` and `home_segments.csv`. These files are not stored in the repository; run the K-means notebook first to create them.

**Outputs:** four CSV files written to `data/processed/`: fact_energyconsumption, fact_homeappliance, dim_home and dim_appliance.

### 1. fact_energyconsumption: one row per reading

- Source: the raw readings (100,000 rows), one row per appliance reading at one home.
- Columns are renamed to short, consistent names (`home_id`, `appliance`, `kwh`, `time`, `date`, `temp`, `season`, `household_size`). No values are changed.
- Used in Power BI for overview charts (consumption by appliance and over time) and for the average kWh per reading of a selected home.
- Note: `household_size` changes within every home, so it is kept as a property of each reading and is not used as a home attribute.

In [5]:
import pandas as pd

# Raw readings
df = pd.read_csv("../data/raw/smart_home_energy_consumption_large.csv")

# Files saved earlier by the K-means notebook
features = pd.read_csv("../data/processed/home_features.csv", index_col="Home ID")
segments = pd.read_csv("../data/processed/home_segments.csv", index_col="Home ID")

appliance_cols = [c for c in features.columns if c not in ["n_readings", "avg_kwh"]]
print(df.shape, features.shape, segments.shape)

(100000, 8) (500, 12) (500, 14)


In [ ]:
fact = df.rename(columns={
    "Home ID": "home_id", "Appliance Type": "appliance",
    "Energy Consumption (kWh)": "kwh", "Time": "time", "Date": "date",
    "Outdoor Temperature (°C)": "temp", "Season": "season",
    "Household Size": "household_size"})
fact.to_csv("../data/processed/fact_energyconsumption.csv", index=False)

### 2. fact_homeappliance: one row per home and appliance
- Source: the home-level feature table (`home_features.csv`), reshaped from 500 rows x 10 appliance columns into a long table of 5,000 rows (500 homes x 10 appliances).
- `avg_kwh`: mean kWh per reading for that home and appliance (a mean, not a total, because homes have between 157 and 245 readings).
- `percentile`: the rank of the home among all 500 homes for that appliance, from 0 to 100 (for example 80 means the home uses at least as much as 80% of homes). A higher value means higher consumption.
- Used in Power BI for the household profile page and the 75th percentile reference line.
- Note: differences between homes are consistent with sampling noise in this dataset, so ranks should be read as position, not as evidence of unusual use.

In [7]:
avg = (features[appliance_cols].reset_index()
       .melt(id_vars="Home ID", var_name="appliance", value_name="avg_kwh"))
pct = (features[appliance_cols].rank(pct=True).mul(100).round(1).reset_index()
       .melt(id_vars="Home ID", var_name="appliance", value_name="percentile"))
home_appliance = avg.merge(pct, on=["Home ID", "appliance"]).rename(columns={"Home ID": "home_id"})
home_appliance.to_csv("../data/processed/fact_homeappliance.csv", index=False)

### 3. dim_home: one row per home
- Source: `home_segments.csv`, which holds the k-means segment of each home (k = 3, `random_state = 0`).
- Columns: `cluster` and `segment_name` (the segment label), `n_readings` (number of readings for the home) and `avg_kwh` (mean kWh per reading across all appliances).
- Used in Power BI as the home selector and to show each home's segment.
- Note: segments are weakly separated and are descriptive only; segment names do not imply behaviour.

In [10]:
dim_home = (segments[["cluster", "segment_name", "n_readings", "avg_kwh"]]
            .reset_index().rename(columns={"Home ID": "home_id"}))

# Display label starting from 1 (the original cluster number is kept)
dim_home["segment_label"] = "Segment " + (dim_home["cluster"] + 1).astype(str)
dim_home.to_csv("../data/processed/dim_home.csv", index=False)

### 4. dim_appliance: one row per appliance
- The list of the 10 appliance types, used as a filter and axis in Power BI.

In [9]:
dim_appliance = pd.DataFrame({"appliance": sorted(appliance_cols)})
dim_appliance.to_csv("../data/processed/dim_appliance.csv", index=False)